# Hyde Park replay — Umwelt Engine walkthrough

This notebook drives the `umwelt` PyO3 extension end-to-end:

1. Synthesise a short sensor log matching the `umwelt.sensor_log.v1` schema (same format the Rust CLI's `umwelt replay` expects).
2. Parse the log, feed it through a small-profile `Pipeline` (4 channels, 64-sample window, 8-bin histogram).
3. Tabulate every Icon / Index / Symbol emission the pipeline produced.

**Prerequisite:** run `maturin develop` from `crates/umwelt-pyo3/` so `import umwelt` resolves to the Rust extension. See `examples/notebooks/README.md`.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import umwelt

print('umwelt extension profile:')
print(f'  CHANNELS    = {umwelt.CHANNELS}')
print(f'  ICON_WINDOW = {umwelt.ICON_WINDOW}')
print(f'  ICON_BINS   = {umwelt.ICON_BINS}')

## 1. Synthesise a sensor log

Every log the Rust CLI consumes is an NDJSON file whose first line is the schema tag `umwelt.sensor_log.v1`. Subsequent lines are `{tick_us, channel, q15}` records. We emit one here to a scratch file so the next cell can treat the pipeline as the CLI would.

In [ ]:
LOG_PATH = Path('hyde_park_sample.ndjson')

def synthesise_log(n_ticks: int = 200, tick_us: int = 10_000) -> list[dict]:
    """Two channels alternating at 0.55 amplitude, 180 degrees out of phase.

    The alternating pattern keeps the icon-layer entropy high; the
    phase offset is small enough to keep the pair inside the 500 ms
    co-occurrence window so Index events fire on every icon pair.
    """
    samples = []
    for i in range(n_ticks):
        q_hi = umwelt.q15_from_f32(0.55 if i % 2 == 0 else -0.55)
        q_lo = umwelt.q15_from_f32(-0.55 if i % 2 == 0 else 0.55)
        t = (i + 1) * tick_us
        samples.append({'tick_us': t, 'channel': 1, 'q15': q_hi})
        samples.append({'tick_us': t + 500, 'channel': 2, 'q15': q_lo})
    return samples

samples = synthesise_log()
with LOG_PATH.open('w') as f:
    f.write('umwelt.sensor_log.v1\n')
    for s in samples:
        f.write(json.dumps(s) + '\n')
print(f'wrote {len(samples)} samples to {LOG_PATH}')

## 2. Parse + replay

Read the NDJSON back (asserting the header), feed every sample into a `Pipeline`, and collect every emitted event.

In [ ]:
def load_log(path: Path) -> list[dict]:
    lines = path.read_text().splitlines()
    assert lines[0] == 'umwelt.sensor_log.v1', f'unexpected header: {lines[0]!r}'
    return [json.loads(line) for line in lines[1:] if line.strip()]

pipeline = umwelt.Pipeline()
events: list[dict] = []
for sample in load_log(LOG_PATH):
    events.extend(pipeline.observe(
        channel=sample['channel'],
        q15=sample['q15'],
        tick_us=sample['tick_us'],
    ))

print(f'pipeline.occupied = {pipeline.occupied}')
print(f'{len(events)} events emitted')
print('first 3 events:')
for ev in events[:3]:
    print(' ', ev)

## 3. Inspect the graph

The PyO3 surface doesn't yet expose a `Graph` view directly (tracking against P7.3+ follow-ups). Instead, aggregate the emission stream by kind and by channel pair — the same signal a graph snapshot would surface.

In [ ]:
by_kind = Counter(ev['kind'] for ev in events)
print('events by kind:')
for kind, count in sorted(by_kind.items()):
    print(f'  {kind:<8} {count}')

icons_per_channel = Counter(ev['channel'] for ev in events if ev['kind'] == 'icon')
print('\nicons per channel:')
for ch, count in sorted(icons_per_channel.items()):
    print(f'  channel {ch}: {count}')

pair_coherence: dict[tuple[int, int], list[float]] = defaultdict(list)
for ev in events:
    if ev['kind'] == 'index':
        a, b = ev['channels']
        pair_coherence[(a, b)].append(ev['coherence'])

print('\nindex coherence by pair:')
for pair, vals in sorted(pair_coherence.items()):
    mean = sum(vals) / len(vals)
    print(f'  {pair}: n={len(vals)}, mean_coherence={mean:.4f}')

## 4. Clean up

Drop the scratch log so re-running the notebook starts clean.

In [ ]:
LOG_PATH.unlink(missing_ok=True)
print('removed scratch log')